# Exploration

## 1) Laden Sie Ihren Datensatz in das Notebook:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Pro Welle eine Datei - Trennzeichen, Gross-/Kleinschreibung und Variablennamen unterscheiden sich!
FILES = {2001: "data/HBSC2001OAed1.0_F4.csv",
         2006: "data/HBSC2006OAed1.0_F1.csv",
         2010: "data/HBSC2010OAed1.0_F4.csv",
         2014: "data/HBSC2014OAed1.1_F1.csv",
         2018: "data/HBSC2018OAed1.1.csv"}       # 2018: Semikolon + BOM

raw = {}
for year, path in FILES.items():
    df = pd.read_csv(path, sep=";" if year == 2018 else ",", encoding="utf-8-sig", low_memory=False)
    df.columns = df.columns.str.lower()            # 2014 hat GROSSGESCHRIEBENE Namen
    raw[year] = df

raw[2001].iloc[:3, :12]

## 2) Kurze Beschreibung des Datensatzes: worum geht es und woher haben Sie den Datensatz (Link)?

Schülerbefragung der WHO-Studie *Health Behaviour in School-aged Children* (HBSC) unter 11-, 13- und
15-Jährigen in Europa und Nordamerika. Pro Zeile eine befragte Person mit rund 120–170 Fragen zu Ernährung, Bewegung,
Medienkonsum, Rauchen, **Alkohol**, Cannabis, Wohlbefinden, Schule, ... . Die Studie läuft alle 4 Jahre. Die Folgenden Jahre waren abrufbar: 2001/02, 2006, 2010, 2014 und 2018 (Diese Daten wurden per Email angefragt auf der offiziellen Seite: https://data-browser.hbsc.org/measure/alcohol-consumption-lifetime-use/#chart).

**Für den Vergleich verwendete Spalten** 

| Thema | Variable | Frage | Skala |
|---|---|---|---|
| Sport | `physact60` | An wie vielen der letzten 7 Tage warst du mind. 60 Min. körperlich aktiv? | 0–7 Tage |
| Alkohol | `drunk` (2018: `drunkltm`) | Schon einmal so viel Alkohol, dass du richtig betrunken warst? | 1 = nie … 5 = >10-mal |
| Alkohol (ergänzend) | `alc30d_2` (2014, 2018) bzw. `drink30d` (2010) | Alkohol an wie vielen Tagen in den letzten 30 Tagen? | 1 = nie … 7 = ≥30 Tage |
| Kontext | `sex`, `age`, `agecat`, `countryno` | Geschlecht, Alter, Altersgruppe (11/13/15), Land | – |

...


## 3. Wie groß ist der Datensatz in Bezug auf Zeilen (Anzahl der Elemente) und Spalten (uni-, bi-, multivariat)? Welche Spalten und Zeilen sind relevant für das Projekt?

*Anmerkung: nicht benötigte Spalten können mittels `drop` entfernt werden.*

In [ ]:
groesse = pd.DataFrame({y: {"Zeilen (Befragte)": d.shape[0], "Spalten": d.shape[1],
                            "Länder/Regionen": d.countryno.nunique()} for y, d in raw.items()}).T
print(groesse.to_string())
print("\nGesamt:", f"{groesse['Zeilen (Befragte)'].sum():,}".replace(",", "."), "Befragte")

gemeinsam = set.intersection(*[set(d.columns) for d in raw.values()])
print(f"\nIn allen 5 Wellen gleich benannte Spalten: {len(gemeinsam)}")
print(sorted(gemeinsam))

      Zeilen (Befragte)  Spalten  Länder/Regionen
2001             162305      127               35
2006             205938      124               41
2010             213595      128               40
2014             214080      170               41
2018             244097      120               47

Gesamt: 1.040.015 Befragte

In allen 5 Wellen gleich benannte Spalten: 44
['age', 'agecat', 'agesex', 'backache', 'beenbullied', 'bodyheight', 'bodyweight', 'breakfastwd', 'breakfastwe', 'bulliedothers', 'contraceptcondom', 'contraceptpill', 'countryno', 'dizzy', 'fatherhome1', 'feellow', 'fight12m', 'fosterhome1', 'hadsex', 'headache', 'health', 'injured12m', 'irritable', 'lifesat', 'likeschool', 'monthbirth', 'motherhome1', 'nervous', 'physact60', 'schoolpressure', 'sex', 'stepfahome1', 'stepmohome1', 'stomachache', 'studaccept', 'studhelpful', 'studtogether', 'talkfather', 'talkmother', 'talkstepfa', 'talkstepmo', 'thinkbody', 'toothbr', 'yearbirth']


## 4. Gibt es fehlende Werte in dem Datensatz? Wenn ja: Wie gehen Sie damit um?

In [ ]:
namen = {"physact60": "physact60", "drunk": "drunk", "drink30d": "drink30d", "alc30d_2": "alc30d_2",
         "alcltm": "alcltm", "alcofreq": "alcofreq", "sampleweights": "sampleweights", "weight": "weight"}
vorhanden = pd.DataFrame({y: {v: (k in d.columns) for k, v in namen.items()} for y, d in raw.items()})
vorhanden.loc["drunk (2018: drunkltm)"] = [("drunk" in d.columns) or ("drunkltm" in d.columns) for d in raw.values()]
print(vorhanden.replace({True: "ja", False: "-"}).to_string())

print("\nDatentypen der Kernvariablen 2018 (alle 'object', weil leere Felder als ' ' gespeichert sind):")
print(raw[2018][["physact60", "drunkltm", "alc30d_2", "agecat", "age"]].dtypes.to_string())
print("\nUngültige Codes 2018 (-99 = 'Missing due to inconsistent answer'):",
      (raw[2018].drunkltm.astype(str).str.strip() == "-99").sum(), "Zeilen bei drunkltm")

                       2001 2006 2010 2014 2018
physact60                ja   ja   ja   ja   ja
drunk                    ja   ja   ja   ja    -
drink30d                  -    -   ja    -    -
alc30d_2                  -    -    -   ja   ja
alcltm                    -    -    -   ja   ja
alcofreq                 ja    -    -    -    -
sampleweights            ja   ja   ja    -    -
weight                    -    -    -    -   ja
drunk (2018: drunkltm)   ja   ja   ja   ja   ja

Datentypen der Kernvariablen 2018 (alle 'object', weil leere Felder als ' ' gespeichert sind):
physact60    object
drunkltm     object
alc30d_2     object
agecat       object
age          object

Ungültige Codes 2018 (-99 = 'Missing due to inconsistent answer'): 697 Zeilen bei drunkltm


**Befund:** Trennzeichen (`,` vs. `;`), Zeichenkodierung (BOM in 2018), Groß-/Kleinschreibung (2014) und Variablennamen
(`drunk` vs. `drunkltm`, `drink30d` vs. `alc30d_2`) unterscheiden sich; 2018 enthält leere Felder als Leerzeichen und den
Sondercode `-99`. Ohne Bereinigung würden 2018-Werte als Text eingelesen; das Alter (Dezimalkomma, z. B. 13,5) ginge beim Umwandeln zu 91 % verloren.

### Task Abstraction

5. Welche Fragen können mit dem gewählten Datensatz untersucht werden?

*Anmerkung: Diese Fragen können in einem domänenspezifischen Format gestellt werden. Hinterfragen Sie hier auch: welche Fragen benötigen eine visuelle Analyse? Nicht nur Bestimmung von Anzahl und Min/Max*

Im Mittelpunkt steht die Frage, wie **körperliche Aktivität** (Tage mit mindestens 60 Minuten Bewegung in der letzten Woche) und **jemals erlebte Betrunkenheit** zusammenhängen. Für alle fünf Erhebungswellen ist dieselbe Art von Vergleich möglich. Der Anteil ‚schon einmal betrunken‘ bezieht sich auf das bisherige Leben, die Bewegung auf die letzte Woche; ein Zusammenhang darf daher nicht als Ursache und Wirkung gelesen werden.

| Nr. | Untersuchungsfrage | Warum visuell untersuchen? |
|---|---|---|
| F1 | Wie verändert sich der Anteil der Jugendlichen, die schon einmal betrunken waren, über die **0 bis 7 Bewegungstage**? Gibt es einen gleichmäßigen Verlauf oder auffällige Werte an den Rändern? | Eine Kurve oder Heatmap zeigt die Form des Zusammenhangs und mögliche Abweichungen, die ein einzelner Durchschnitt verdeckt. |
| F2 | Unterscheidet sich dieser Verlauf zwischen **11-, 13- und 15-Jährigen** sowie zwischen Jungen und Mädchen? | Mehrere Kurven oder kleine, gleich skalierte Diagramme machen Unterschiede zwischen Gruppen und mögliche Wechselwirkungen sichtbar. |
| F3 | Welche **Länder und Regionen** liegen 2018 bei durchschnittlicher Bewegung und beim Anteil ‚schon einmal betrunken‘ nahe beieinander, und welche weichen deutlich ab? | Ein Streudiagramm zeigt gleichzeitig beide Merkmale, die Streuung und auffällige Länder. Die Fallzahl kann zusätzlich dargestellt werden. |
| F4 | Wie verändern sich Bewegung und der Anteil ‚schon einmal betrunken‘ von **2001 bis 2018**? Verlaufen die Entwicklungen in den Ländern ähnlich? | Zeitreihen pro Merkmal und Land zeigen Richtung, Stärke und Unterschiede der Veränderungen über fünf Wellen. Für Vergleiche werden nur Länder mit Daten in allen Wellen und gleiche Altersgruppen betrachtet. |

Für alle Fragen werden nur gültige Antworten auf die jeweils benötigten Merkmale verwendet. Länder- und Gruppenvergleiche berücksichtigen die jeweilige Fallzahl; die wiederholten Befragungen verfolgen nicht dieselben Personen über die Zeit.

6. Welche abstrakten Aufgaben stecken in den Fragen? *(z.B. Finden von Ausreißern, Vergleich von mehreren Werten)*

| Frage | Abstrakte Aufgabe | Benötigte Operation |
|---|---|---|
| F1 | **Zusammenhang erkennen und Verteilung vergleichen:** Verändert sich ein Anteil über die geordnete Anzahl der Bewegungstage? | Nach Bewegungstagen gruppieren, den Anteil mit mindestens einer Betrunkenheit berechnen und den Verlauf auf nicht lineare Muster oder Ausreißer prüfen. |
| F2 | **Teilgruppen vergleichen:** Bleibt das Muster gleich, wenn nach Alter und Geschlecht unterschieden wird? | Die Anteile zusätzlich nach Altersgruppe und Geschlecht aufschlüsseln und Verläufe auf gleicher Skala vergleichen. |
| F3 | **Objekte vergleichen, Gruppen und Ausreißer finden:** Welche Länder haben ähnliche oder ungewöhnliche Kombinationen beider Merkmale? | Je Land den mittleren Bewegungswert, den Anteil ‚schon einmal betrunken‘ und die Fallzahl aggregieren; Positionen und Abstände im Streudiagramm vergleichen. |
| F4 | **Zeitliche Trends und Veränderungen vergleichen:** Wo steigen, fallen oder schwanken die beiden Merkmale? | Nach Erhebungsjahr und Land aggregieren, auf vergleichbare Länder und Altersgruppen filtern und die zeitlichen Verläufe gegenüberstellen. |

Die Analyse wechselt damit von **Vergleichen nach individuellen Merkmalen** (F1/F2) zu **Länder- und Zeitvergleichen** (F3/F4). Bei Anteilen ist der Nenner jeweils die Zahl der Befragten mit gültigen Angaben in der betrachteten Gruppe.

### Datentransformation

7. Inwiefern ist es notwendig den Datensatz für die Beantwortung der Fragen anzupassen (z.B. Verknüpfungen, Filterung, Änderung des Detailgrads)? Führen Sie diese Datentransformationen in Python durch und binden den finalen Datensatz nochmal tabellarisch ein, da sich die weiteren Aufgaben darauf beziehen.

**Notwendige Anpassungen für F1–F4:**

1. **Wellen zusammenführen:** Die fünf CSV-Dateien enthalten dieselbe Art von Beobachtung (eine befragte Person je Zeile). Sie werden untereinander angehängt; eine Verknüpfung über Personen-IDs wäre falsch, weil in jeder Welle andere Jugendliche befragt wurden. `drunk` heißt 2018 `drunkltm`.
2. **Werte vereinheitlichen:** Leerzeichen, leere Felder und der Sondercode `-99` werden als fehlend behandelt. Nur gültige Codes für Bewegung (0–7), Betrunkenheit (1–5), Altersgruppe (1–3) und Geschlecht (1–2) bleiben erhalten. Länder-Codes erhalten lesbare Namen.
3. **Auf die Fragestellung filtern:** Für einen konsistenten Vergleich werden nur Zeilen mit gültigen Angaben zu allen vier Kernmerkmalen sowie einem bekannten Land verwendet. Aus der fünfstufigen Betrunkenheitsskala wird zusätzlich `drunk_ever` abgeleitet (0 = nie, 1 = mindestens einmal). Fehlende Angaben werden dabei nicht als ‚nie‘ gewertet.
4. **Detailgrad anpassen:** `hbsc_final` bleibt auf Personenebene und ermöglicht F1/F2. Für F3/F4 entstehen daraus Tabellen mit Fallzahl, mittleren Bewegungstagen und Betrunkenheitsanteil je Land/Jahr beziehungsweise je Land/Jahr/Altersgruppe. Für F4 werden nur Land-Altersgruppen mit gültigen Daten in allen fünf Wellen verwendet. Die Anteile und Mittelwerte sind ungewichtet.

Die folgende Tabelle zeigt die ersten Zeilen von `hbsc_final`. Der vollständige Datensatz bleibt als DataFrame im Notebook verfügbar; eine Ausgabe aller rund eine Million Zeilen wäre nicht sinnvoll.

In [ ]:
# Die Dateien sind oben bereits als raw[Erhebungsjahr] geladen; Spaltennamen sind kleingeschrieben.
COUNTRY_NAMES = {
    8000: 'Albanien', 31000: 'Aserbaidschan', 40000: 'Österreich', 51000: 'Armenien',
    56001: 'Belgien (flämisch)', 56002: 'Belgien (französisch)', 100000: 'Bulgarien',
    124000: 'Kanada', 191000: 'Kroatien', 203000: 'Tschechien', 208000: 'Dänemark',
    233000: 'Estland', 246000: 'Finnland', 250000: 'Frankreich', 268000: 'Georgien',
    276000: 'Deutschland', 300000: 'Griechenland', 304000: 'Grönland', 348000: 'Ungarn',
    352000: 'Island', 372000: 'Irland', 376000: 'Israel', 380000: 'Italien',
    398000: 'Kasachstan', 428000: 'Lettland', 440000: 'Litauen', 442000: 'Luxemburg',
    470000: 'Malta', 498000: 'Moldau', 528000: 'Niederlande', 578000: 'Norwegen',
    616000: 'Polen', 620000: 'Portugal', 642000: 'Rumänien', 643000: 'Russland',
    688000: 'Serbien', 703000: 'Slowakei', 705000: 'Slowenien', 724000: 'Spanien',
    752000: 'Schweden', 756000: 'Schweiz', 792000: 'Türkei', 804000: 'Ukraine',
    807000: 'Nordmazedonien', 826001: 'England', 826002: 'Schottland',
    826003: 'Wales', 840000: 'USA'
}

def als_zahl(spalte):
    # 2018 enthält Leerzeichen und teils Dezimalkommas; -99 bedeutet ungültige Antwort.
    zahl = pd.to_numeric(spalte.astype(str).str.strip().str.replace(',', '.', regex=False),
                         errors='coerce')
    return zahl.replace(-99, np.nan)

teile = []
for welle, daten in raw.items():
    alkoholspalte = 'drunkltm' if welle == 2018 else 'drunk'
    teile.append(pd.DataFrame({
        'welle': welle,
        'country_code': als_zahl(daten['countryno']),
        'sex_code': als_zahl(daten['sex']),
        'agecat': als_zahl(daten['agecat']),
        'sport_days': als_zahl(daten['physact60']),
        'drunk_lifetime': als_zahl(daten[alkoholspalte])
    }))

hbsc_final = pd.concat(teile, ignore_index=True)
for spalte, codes in {'sex_code': [1, 2], 'agecat': [1, 2, 3],
                      'sport_days': range(8), 'drunk_lifetime': range(1, 6)}.items():
    hbsc_final[spalte] = hbsc_final[spalte].where(hbsc_final[spalte].isin(codes))
hbsc_final['country'] = hbsc_final['country_code'].map(COUNTRY_NAMES)
hbsc_final = hbsc_final.dropna(subset=[
    'country', 'sex_code', 'agecat', 'sport_days', 'drunk_lifetime'
]).copy()
hbsc_final['sex'] = hbsc_final['sex_code'].map({1: 'Junge', 2: 'Mädchen'}).astype('category')
hbsc_final['age_group'] = pd.Categorical(
    hbsc_final['agecat'].map({1: '11 Jahre', 2: '13 Jahre', 3: '15 Jahre'}),
    categories=['11 Jahre', '13 Jahre', '15 Jahre'], ordered=True
)
hbsc_final['drunk_ever'] = (hbsc_final['drunk_lifetime'] > 1).astype('int8')
hbsc_final['welle'] = hbsc_final['welle'].astype('int16')
hbsc_final['country_code'] = hbsc_final['country_code'].astype('int32')
hbsc_final['sport_days'] = hbsc_final['sport_days'].astype('int8')
hbsc_final['drunk_lifetime'] = hbsc_final['drunk_lifetime'].astype('int8')
hbsc_final = hbsc_final[[
    'welle', 'country_code', 'country', 'sex', 'age_group',
    'sport_days', 'drunk_lifetime', 'drunk_ever'
]].reset_index(drop=True)
hbsc_final['country'] = hbsc_final['country'].astype('category')

# Aggregation für Länder- und Zeitvergleiche; n zeigt die jeweilige Fallzahl.
def nach_gruppen(gruppen):
    tabelle = (hbsc_final.groupby(gruppen, observed=True)
               .agg(n=('drunk_ever', 'size'),
                    sport_mean=('sport_days', 'mean'),
                    drunk_pct=('drunk_ever', 'mean'))
               .reset_index())
    tabelle['drunk_pct'] *= 100
    return tabelle

hbsc_country_year = nach_gruppen(['welle', 'country'])  # F3: 2018 auswählen
hbsc_country_age_year = nach_gruppen(['welle', 'country', 'age_group'])
anzahl_wellen = hbsc_country_age_year.groupby(
    ['country', 'age_group'], observed=True)['welle'].transform('nunique')
hbsc_trend = hbsc_country_age_year.loc[anzahl_wellen.eq(len(FILES))].copy()  # F4

print(f'Finaler Personendatensatz: {len(hbsc_final):,} Zeilen, {hbsc_final.shape[1]} Spalten')
print(f'Ausgeschlossene Zeilen mit fehlenden/ungültigen Kernangaben: {sum(len(d) for d in raw.values()) - len(hbsc_final):,}')
display(hbsc_final.head(10))
print(f'Trendtabelle: {len(hbsc_trend):,} Land-Altersgruppen-Wellen mit vollständiger Zeitreihe')
display(hbsc_trend.head(10))

Finaler Personendatensatz: 948,095 Zeilen, 8 Spalten
Ausgeschlossene Zeilen mit fehlenden/ungültigen Kernangaben: 91,920


,welle,country_code,country,sex,age_group,sport_days,drunk_lifetime,drunk_ever
0,2001,40000,Österreich,Mädchen,11 Jahre,1,1,0
1,2001,40000,Österreich,Mädchen,11 Jahre,2,1,0
2,2001,40000,Österreich,Junge,11 Jahre,5,1,0
3,2001,40000,Österreich,Junge,11 Jahre,7,1,0
4,2001,40000,Österreich,Mädchen,11 Jahre,6,1,0
5,2001,40000,Österreich,Junge,11 Jahre,2,1,0
6,2001,40000,Österreich,Mädchen,11 Jahre,7,1,0
7,2001,40000,Österreich,Junge,11 Jahre,5,1,0
8,2001,40000,Österreich,Junge,11 Jahre,4,1,0
9,2001,40000,Österreich,Junge,11 Jahre,4,1,0


Trendtabelle: 425 Land-Altersgruppen-Wellen mit vollständiger Zeitreihe


,welle,country,age_group,n,sport_mean,drunk_pct
0,2001,Belgien (flämisch),11 Jahre,2043,3.396476,11.845326
1,2001,Belgien (flämisch),13 Jahre,2028,3.122288,22.287968
2,2001,Belgien (flämisch),15 Jahre,1954,3.061412,50.153531
3,2001,Deutschland,11 Jahre,1947,3.860298,8.217771
4,2001,Deutschland,13 Jahre,1740,3.693678,25.977011
5,2001,Deutschland,15 Jahre,1681,3.500892,56.811422
6,2001,Dänemark,11 Jahre,1510,4.052980,11.721854
7,2001,Dänemark,13 Jahre,1456,3.777473,33.722527
8,2001,Dänemark,15 Jahre,1313,3.533892,78.065499
9,2001,England,11 Jahre,2146,4.466449,24.603914


---

### Data Abstraction

8. Welche Attributtypen können den einzelnen Spalten zugewiesen werden? Geben Sie für ordinale und nominale Daten an, wieviele eindeutige Klassen pro Spalte enthalten sind! Geben Sie für quantitative Daten den Wertebereich an (Minimum und Maximum) (in Python mit min/max bzw. unique())!

9. Stecken weitere semantische Datenstrukturen in dem Datensatz (z.B. hierarchisch, geografisch, temporal), die für die Visualisierungen nützlich sein können?

...

---

### Visuelle Exploration des Datensatzes

10. Explorieren Sie die Eigenschaften des Datensatzes mit mindestens zwei interaktiven (verschiedene) Visualisierungen pro Teammitglied! (z.B. Verteilungen oder Korrelationen untersuchen)

11. Was haben Sie über die Eigenschaften der Daten erfahren? Welche Muster haben Sie erkannt? Welche Insights sind interessant für die Explanation Phase? (dies kann auch direkt unter jeder Visualisierung beantwortet werden)

...